<a href="https://colab.research.google.com/github/gbrixi/minerva/blob/main/examples/notebooks/finetune_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minerva — LoRA finetuning (Colab)

Parameter-efficient **LoRA** masked-language-model finetuning of Minerva on your own genome
(a GenBank `.gb`). Trains ~1% of the parameters, so it fits a single Colab GPU.

Reuses the training core from `minerva.finetuning` (shared with `scripts/finetune.py`).
Optionally shows the **contact map before vs after** finetuning.

**Fill the form, then `Runtime` → `Run all`.**

In [ ]:
#@title Setup — install &amp; import { display-mode: "form" }
import importlib.util, os, sys, types, collections.abc as _abc
IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/gbrixi/minerva.git"  #@param {type:"string"}
if IN_COLAB and not os.path.isfile("MINERVA_READY"):
    !pip -q install biopython peft datasets accelerate
    if not os.path.isdir("minerva"):
        !git clone -q $REPO_URL
    open("MINERVA_READY", "w").close()
# Shim for environments with an old deepspeed that imports the removed torch._six.
if "torch._six" not in sys.modules:
    _six = types.ModuleType("torch._six"); _six.inf = float("inf"); _six.nan = float("nan")
    _six.container_abcs = _abc; _six.string_classes = (str, bytes); _six.int_classes = (int,)
    sys.modules["torch._six"] = _six
for _p in (".", "..", "minerva", os.path.expanduser("~/minerva")):
    if os.path.isdir(os.path.join(_p, "minerva")):
        sys.path.insert(0, os.path.abspath(_p)); break

# Drop any already-imported Minerva modules/objects so rerunning setup after a
# code update cannot keep stale code or an old fp16-loaded model in memory.
for _name in list(sys.modules):
    if _name == "minerva" or _name.startswith("minerva."):
        sys.modules.pop(_name, None)
for _old in ("model", "tokenizer", "lora_model", "trainer"):
    globals().pop(_old, None)
import torch
from transformers import AutoTokenizer, TrainingArguments, DataCollatorForLanguageModeling
from minerva.modeling_minerva import MinervaForMaskedLM
from minerva.finetuning import build_block_dataset, apply_lora, MinervaTrainer
from minerva.data import extract_and_tokenize_gb
import minerva.modeling_minerva as _mm

# Force the torch SDPA path when flash-attn is not explicitly requested.
# This avoids Colab/T4 using a preinstalled or half-installed flash-attn build.
if not globals().get("install_flash_attn", False):
    _mm._HAS_FLASH = False
    _mm._HAS_FLASH_ROTARY = False

import inspect
_rotary_forward_src = inspect.getsource(_mm.RotaryEmbedding.forward)
if 'q.device.type != "cuda"' not in _rotary_forward_src or 'return self.apply_torch(q, k)' not in _rotary_forward_src:
    raise RuntimeError("Imported Minerva lacks the non-flash rotary fallback. Restart and import patched code.")
_rmsnorm_src = inspect.getsource(_mm.rmsnorm_func)
if 'hidden_states.float()' not in _rmsnorm_src:
    raise RuntimeError("Imported Minerva has fp16-unsafe RMSNorm. Restart and import patched code.")
_rms_x = torch.full((1, 4), 322.0, dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
_rms_w = torch.ones(4, dtype=_rms_x.dtype)
_rms_y = _mm.rmsnorm_func(_rms_x, _rms_w, torch.tensor(1e-5))
if (not torch.isfinite(_rms_y).all()) or bool(torch.all(_rms_y == 0)):
    raise RuntimeError("Imported Minerva failed the fp16 RMSNorm runtime check. Restart runtime and rerun setup.")
print("ready | colab:", IN_COLAB, "| flash-attn enabled:", _mm._HAS_FLASH)

In [ ]:
#@title Hugging Face login (private base model) { display-mode: "form" }
#@markdown Needs read access to the base model + an HF token. Colab Secret `HF_TOKEN`,
#@markdown environment variable `HF_TOKEN`, or a cached Hugging Face token will be reused.
from huggingface_hub import get_token, login
_tok = None
try:
    from google.colab import userdata; _tok = userdata.get("HF_TOKEN")
except Exception:
    _tok = os.environ.get("HF_TOKEN")
_tok = _tok or get_token()
if _tok:
    login(token=_tok, new_session=False)
else:
    login(new_session=False)

In [ ]:
#@title 1. Data & finetuning settings { display-mode: "form", run: "auto" }
base_model = "minerva-1"      #@param {type:"string"}
base_model = resolve_minerva_model(base_model)
data       = "twoayggay"             #@param ["ug27", "twoayggay", "upload your own"]
#@markdown LoRA settings matching the historical `lora1` fine-tune launch.
lora_r       = 1     #@param {type:"integer"}
lora_alpha   = 2     #@param {type:"integer"}
lora_dropout = 0.05  #@param {type:"number"}
#@markdown Training. `block_size` is the token chunk length; it corresponds to `--max_seq_length` in `scripts/finetune.py`.
block_size    = 8192   #@param {type:"integer"}
learning_rate = 1e-4   #@param {type:"number"}
epochs        = 3      #@param {type:"number"}
max_steps     = 12000  #@param {type:"integer"}
batch_size    = 1      #@param {type:"integer"}
grad_accum_steps = 2   #@param {type:"integer"}
gradient_checkpointing = True  #@param {type:"boolean"}
mask_ratio    = 0.15   #@param {type:"number"}
token_type_upweighting = True  #@param {type:"boolean"}
precision     = "fp16"  #@param ["fp16", "bf16", "fp32"]
#@markdown Contact-map preview can be expensive at 8192 tokens; turn it on for short exploratory runs.
show_before_after = False  #@param {type:"boolean"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    precision = "fp32"
DTYPE = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[precision]
print("device:", DEVICE, "| precision:", precision)


In [ ]:
#@title (optional) Upload your own GenBank { display-mode: "form" }
UPLOADED = None
if IN_COLAB and data == "upload your own":
    from google.colab import files
    up = files.upload()
    if up:
        UPLOADED = list(up.keys())[0]; print("uploaded:", UPLOADED)

In [ ]:
# Resolve the GenBank, load base model + tokenizer.
def _find(name):
    for base in ("data", "examples/data", "notebooks/data", "minerva/notebooks/data", os.path.expanduser("~/minerva/examples/data")):
        if os.path.exists(os.path.join(base, name)):
            return os.path.join(base, name)
    raise FileNotFoundError(name)
GB = UPLOADED if data == "upload your own" else _find(
    "UG27_systems.gb" if data == "ug27" else "TwoAYGGAY_Pseudomonas_fluorescens_SBW25.gb")

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = MinervaForMaskedLM.from_pretrained(base_model, torch_dtype=DTYPE).to("cuda").eval()
print("base model loaded | data:", GB)

In [ ]:
# Optional: contact map BEFORE finetuning (compute on the base model first).
rgb_before = None
if show_before_after:
    from minerva.visualization import publication_head_contacts_rgb
    _rec = extract_and_tokenize_gb(GB, use_existing_translations=True)[0]
    _seq = _rec["sequence"]
    _toks = tokenizer.convert_ids_to_tokens(tokenizer.encode(_seq))
    _w = (0, min(block_size, len(_toks)))
    def _overlay(mdl):
        pr = mdl.predict_contacts(sequence=_seq, tokenizer=tokenizer,
                                  head_names=["base_pairing", "repeat", "protein"],
                                  seed_start=_w[0], seed_end=_w[1])["predictions"]
        ch = {h: pr[h].float().cpu().numpy() for h in ["base_pairing", "repeat", "protein"]}
        print("preview head ranges:", {k: (float(np.nanmin(v)), float(np.nanmax(v))) for k, v in ch.items()})
        return publication_head_contacts_rgb(ch, tokens=_toks[_w[0]:_w[1]])
    with torch.no_grad():
        rgb_before = _overlay(model)
    print("computed 'before' contact map")


In [ ]:
# Build fixed-length blocks, apply LoRA, train.
ds = build_block_dataset(GB, tokenizer, block_size=block_size)
print(ds)
if len(ds["train"]) == 0:
    raise ValueError(f"No training chunks were produced. Lower block_size below {block_size} or use longer loci.")

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=mask_ratio)
_probe = collator([ds["train"][0]])
_n_masked = int((_probe["labels"] != -100).sum().item())
print("masked labels in first batch:", _n_masked)
if _n_masked == 0:
    raise ValueError("The MLM collator produced zero labels for the first batch; loss would be undefined/zero.")

if gradient_checkpointing and hasattr(model, "minerva") and hasattr(model.minerva, "encoder"):
    model.minerva.encoder.gradient_checkpointing = True
    print("Set gradient_checkpointing=True on encoder")

lora_model = apply_lora(model, r=lora_r, alpha=lora_alpha, dropout=lora_dropout)
lora_model.print_trainable_parameters()

vocab = tokenizer.get_vocab()
nuc = torch.tensor([vocab[c] for c in "atgcn" if c in vocab])
aa  = torch.tensor([vocab[c] for c in "ACDEFGHIKLMNPQRSTVWY" if c in vocab])

_targs_kwargs = dict(
    output_dir="minerva_ft", per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum_steps,
    learning_rate=learning_rate, num_train_epochs=epochs,
    fp16=(DEVICE == "cuda" and precision == "fp16"),
    bf16=(DEVICE == "cuda" and precision == "bf16"),
    logging_steps=10, save_strategy="no", report_to=[], remove_unused_columns=False)
if max_steps and max_steps > 0:
    _targs_kwargs["max_steps"] = max_steps
targs = TrainingArguments(**_targs_kwargs)
trainer = MinervaTrainer(
    model=lora_model, nuc_tokens=nuc, aa_tokens=aa, token_type_upweighting=token_type_upweighting,
    args=targs, train_dataset=ds["train"], eval_dataset=ds.get("validation"),
    data_collator=collator)
trainer.train()
lora_model.save_pretrained("minerva_lora_adapter")
print("saved LoRA adapter -> minerva_lora_adapter/")


In [ ]:
# Optional: contact map AFTER finetuning (merge LoRA, re-run the viewer) side by side.
if show_before_after and rgb_before is not None:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from minerva.visualization import render_publication_panel, PUBLICATION_DATASET_COLORS
    merged = lora_model.merge_and_unload().eval()
    with torch.no_grad():
        rgb_after = _overlay(merged)
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 5.2))
    for ax, rgb, ttl in zip(axes, [rgb_before, rgb_after], ["before", f"after ({epochs} ep)"]):
        render_publication_panel(ax, rgb, ttl, show_yticks=(ax is axes[0]))
    handles = [
        mpatches.Patch(facecolor=PUBLICATION_DATASET_COLORS["pdb"], edgecolor="black", linewidth=0.4, label="PDB"),
        mpatches.Patch(facecolor=PUBLICATION_DATASET_COLORS["rna"], edgecolor="black", linewidth=0.4, label="RNA"),
        mpatches.Patch(facecolor=PUBLICATION_DATASET_COLORS["repeat"], edgecolor="black", linewidth=0.4, label="Repeats"),
    ]
    leg = axes[1].legend(handles=handles, loc="upper right", fontsize=8,
                         frameon=True, framealpha=0.9, edgecolor="black", fancybox=False)
    leg.get_frame().set_linewidth(0.6)
    plt.suptitle("Minerva contacts — LoRA finetuning effect", fontsize=12, fontstyle="italic")
    plt.tight_layout(); plt.savefig("finetune_before_after.pdf", dpi=600, bbox_inches="tight", facecolor="white"); plt.show()
    print("saved finetune_before_after.pdf")


---
**Notes**
- This is **LoRA** finetuning (adapter saved to `minerva_lora_adapter/`). Load it later with
  `peft.PeftModel.from_pretrained(base, "minerva_lora_adapter")`, or `merge_and_unload()` for a plain model.
- Defaults mirror the historical `lora1` launch: rank 1, alpha 2, batch size 1, gradient accumulation 2, gradient checkpointing on, token-type upweighting on, and 8192-token chunks.
- `block_size` is the token chunk length used for training examples; in `scripts/finetune.py` this is `--max_seq_length`.
- The shipped examples are small demos. Upload a larger genome for a real fine-tune.
- Contact-map preview is off by default because full 8192-token maps are expensive on Colab.
- By default this notebook uses `minerva-1/` from the private Drive bundle, so no Hugging Face token or GitHub clone is required.
